In [14]:
!pip install datasets evaluate rouge_score accelerate

In [15]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig, GPTQConfig, StoppingCriteria, StoppingCriteriaList, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import nltk
from random import randint
import json

In [16]:
!pip -q kaggle

ERROR: unknown command "kaggle"


In [17]:
# Загрузка файла Kaggle
from google.colab import files
files.upload()

# Настройка Kaggle API
!rm -r ~/.kaggle
!mkdir ~/.kaggle
!mv ./kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Загрузка и распаковка набора данных
!kaggle datasets download -d stanfordu/stanford-question-answering-dataset
zip_ref = zipfile.ZipFile('/content/stanford-question-answering-dataset.zip', 'r')
zip_ref.extractall('/content')
zip_ref.close()


mv: cannot stat './kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
Dataset URL: https://www.kaggle.com/datasets/stanfordu/stanford-question-answering-dataset
License(s): CC-BY-SA-4.0
stanford-question-answering-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)


In [6]:
import zipfile
zip_ref = zipfile.ZipFile('/content/stanford-question-answering-dataset.zip', 'r')
zip_ref.extractall('/content')
zip_ref.close()

In [18]:
import os
import json

from datasets import Dataset
from transformers import AutoTokenizer
from transformers import DefaultDataCollator
from transformers import AutoModelForQuestionAnswering, TrainingArguments, Trainer

In [19]:
import warnings
warnings.filterwarnings("ignore")

In [20]:
with open('dev-v1.1.json', 'r') as _f:
    dev = json.load(_f)

with open('train-v1.1.json', 'r') as _f:
    train = json.load(_f)

In [21]:
def create_dataset(data):
    contexts, questions, answers = [], [], []
    for item in data['data']:
        for paragraph in item['paragraphs']:
            context = paragraph['context']
            for qa in paragraph['qas']:
                question = qa['question']
                for answer in qa['answers']:
                    contexts.append(context)
                    questions.append(question)
                    answers.append({'text': answer['text'], 'answer_start': answer['answer_start']})
    return Dataset.from_dict({'context': contexts, 'question': questions, 'answers': answers})

In [29]:
TRAIN = create_dataset(train)
TEST = create_dataset(dev)

TRAIN = TRAIN.train_test_split(test_size=0.2, seed=4444)

TRAIN

DatasetDict({
    train: Dataset({
        features: ['context', 'question', 'answers'],
        num_rows: 70079
    })
    test: Dataset({
        features: ['context', 'question', 'answers'],
        num_rows: 17520
    })
})

In [25]:
TRAIN

DatasetDict({
    train: Dataset({
        features: ['context', 'question', 'answers'],
        num_rows: 70079
    })
    test: Dataset({
        features: ['context', 'question', 'answers'],
        num_rows: 17520
    })
})

In [31]:
tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")
model = AutoModelForQuestionAnswering.from_pretrained("distilbert/distilbert-base-uncased")
data_collator = DefaultDataCollator()

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForQuestionAnswering were not initialized from the model checkpoint at distilbert/distilbert-base-uncased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [41]:
def preprocess_function(examples):
    # Подготовка вопросов и токенизация с извлечением необходимых полей
    inputs = tokenizer(
        [q.strip() for q in examples["question"]],  # Удаляем лишние пробелы
        examples["context"],
        max_length=500,
        truncation="only_second",
        return_offsets_mapping=True,
        padding="max_length",
    )
    offset_mapping = inputs.pop("offset_mapping")
    answers = examples["answers"]
    sequence_ids_list = [inputs.sequence_ids(i) for i in range(len(offset_mapping))]

    # Формируем позиции начала и конца ответов
    start_positions, end_positions = [], []
    for offsets, answer, sequence_ids in zip(offset_mapping, answers, sequence_ids_list):
        start_char, end_char = answer["answer_start"], answer["answer_start"] + len(answer["text"])
        context_start = sequence_ids.index(1)
        context_end = len(sequence_ids) - sequence_ids[::-1].index(1) - 1

        # Проверка, находится ли ответ в пределах контекста
        if offsets[context_start][0] > end_char or offsets[context_end][1] < start_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            # Определение начала и конца ответов
            start_positions.append(
                next((idx for idx in range(context_start, context_end + 1) if offsets[idx][0] >= start_char), 0) # Add a default value of 0
            )
            end_positions.append(
                next((idx for idx in range(context_end, context_start - 1, -1) if offsets[idx][1] <= end_char), 0) # Add a default value of 0
            )

    inputs.update({"start_positions": start_positions, "end_positions": end_positions})
    return inputs

In [42]:
TRAIN = create_dataset(train)
TEST = create_dataset(dev)

train_test_split_output = TRAIN.train_test_split(test_size=0.2, seed=4444)
train_dataset = train_test_split_output["train"]
test_dataset = train_test_split_output["test"]

TRAIN

Dataset({
    features: ['context', 'question', 'answers'],
    num_rows: 87599
})

In [43]:
training_args = TrainingArguments(
    output_dir="my_model",
    evaluation_strategy="epoch",
    save_strategy='epoch',
    learning_rate=1e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    push_to_hub=False,
    report_to = 'none',
    load_best_model_at_end = True,
    overwrite_output_dir = True,
    metric_for_best_model= "eval_loss",
    greater_is_better= False
)

In [44]:
train_tokenized = train_dataset.map(preprocess_function, batched=True)
test_tokenized = test_dataset.map(preprocess_function, batched=True)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

Map:   0%|          | 0/70079 [00:00<?, ? examples/s]

Map:   0%|          | 0/17520 [00:00<?, ? examples/s]

In [50]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,1.416600,1.236312
2,1.173800,1.180264


Epoch,Training Loss,Validation Loss


TrainOutput(global_step=8760, training_loss=1.4752921640056453, metrics={'train_runtime': 7179.9046, 'train_samples_per_second': 19.521, 'train_steps_per_second': 1.22, 'total_flos': 1.7952297344436e+16, 'train_loss': 1.4752921640056453, 'epoch': 2.0})

In [51]:
trainer.save_model('working/final_model')

In [52]:
trainer.evaluate(eval_dataset=test_tokenized)

{'eval_loss': 1.180263876914978,
 'eval_runtime': 260.2325,
 'eval_samples_per_second': 67.324,
 'eval_steps_per_second': 4.208,
 'epoch': 2.0}

In [53]:
from transformers import pipeline

question_answerer = pipeline("question-answering", model="working/final_model")

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


In [54]:
context = """India, a captivating mosaic of cultures, religions, and traditions, enthralls with its unparalleled diversity. Its ancient civilization, steeped in history and spanning millennia, has profoundly influenced global heritage. From the vibrant markets of Delhi to the idyllic shores of Goa, India's landscapes boast a breathtaking array of sights and experiences. Chennai, a city in the state of tamil nadu is expected to be the next tech hub in India following bangalore and mumbai. Yet, amidst the splendor, persistent challenges such as poverty and environmental degradation demand attention. With 28 states and 8 union territories, India stands as a testament to unity in diversity. In this vast expanse, Mumbai, the bustling metropolis pulsating with life, claims the title of the largest city by population. As India strides confidently into the future, it remains a land of contrasts, blending tradition with modernity, and embracing its complexities with resilience and grace."""

question_1 = 'Which city is the future tech hub?'
question_2 = 'Which city has the most population?'
question_3 = 'How many states and union territories are there in india?'

print(question_answerer(question=question_1, context=context))
print(question_answerer(question=question_2, context=context))
print(question_answerer(question=question_3, context=context))

{'score': 0.47779572010040283, 'start': 360, 'end': 367, 'answer': 'Chennai'}
{'score': 0.5230209231376648, 'start': 705, 'end': 711, 'answer': 'Mumbai'}
{'score': 0.3227119743824005, 'start': 597, 'end': 612, 'answer': '28 states and 8'}
